In [2]:
#!/usr/bin/env python3
"""AURORA 199-patient ER timing audit.

Recreates the ER-retained (n=179) versus ER-loss (n=20) cohort and audits
the first cfDNA/Griffin draw relative to metastatic diagnosis and the first
informative metastatic ER assessment.
"""
from pathlib import Path
import numpy as np
import pandas as pd


# =============================================================================
# USER CONFIGURATION
# =============================================================================
AURORA_BASE = Path("/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA")
CLINICAL_XLSX = AURORA_BASE / "Clinical_data" / "AUP_ClinicalData_20260609.xlsx"
SPLIT_TSV = (
    AURORA_BASE / "Griffin" / "Downstream" / "AURORA_Griffin_Heldout_Test_ER"
    / "Cohort_split.tsv"
)
FIRSTDRAW_TSV = (
    AURORA_BASE / "Griffin" / "Downstream"
    / "AURORA_Griffin_Met_ER_FirstDraw_TFBS"
    / "First_draw_blood_aligned_metastatic_ER_dataset.tsv"
)
OUT_DIR = (
    AURORA_BASE / "Griffin" / "Downstream" / "ER_phenotype"
    / "199_patient_ER_timing_audit"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_TSV = OUT_DIR / "AURORA_199_patient_ER_timing_audit.tsv"
SUMMARY_TSV = OUT_DIR / "AURORA_199_patient_ER_timing_summary.tsv"
AUDIT_XLSX = OUT_DIR / "AURORA_199_patient_ER_timing_audit.xlsx"


def normalize_patient_id(value):
    """Standardize IDs that may have been read as Excel numeric values."""
    if pd.isna(value):
        return np.nan
    value_str = str(value).strip()
    try:
        value_float = float(value_str)
        if value_float.is_integer() and value_str.replace(".0", "").isdigit():
            return str(int(value_float))
    except (TypeError, ValueError):
        pass
    return value_str


def normalize_er(value):
    """Map common ER representations to Positive/Negative; otherwise missing."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    if value in {"positive", "pos", "+", "er+"}:
        return "Positive"
    if value in {"negative", "neg", "-", "er-"}:
        return "Negative"
    return np.nan


def find_clinical_sheet(path):
    required = {
        "patient_id", "Category", "Days_From_Primary_Diagnosis", "ER_Status", "Tissue_Site"
    }
    with pd.ExcelFile(path) as xls:
        for sheet_name in xls.sheet_names:
            columns = set(pd.read_excel(path, sheet_name=sheet_name, nrows=0).columns)
            if required.issubset(columns):
                return sheet_name
    raise ValueError(f"No clinical worksheet contains all required columns: {sorted(required)}")


def main():
    for path in (CLINICAL_XLSX, SPLIT_TSV, FIRSTDRAW_TSV):
        if not path.exists():
            raise FileNotFoundError(f"Missing input file: {path}")

    # Fixed receptor-evolution cohort: primary ER+ with informative metastatic ER.
    split = pd.read_csv(SPLIT_TSV, sep="\t", dtype={"patient_id": str})
    required_split = {"patient_id", "sample", "Primary_tumor_ER"}
    missing = required_split - set(split.columns)
    if missing:
        raise ValueError(f"Cohort_split.tsv is missing: {sorted(missing)}")
    # Current AURORA split files name the endpoint Earliest_metastatic_ER.
    # Older derived versions used Metastatic_ER.  Internally we use the latter
    # neutral name so every downstream audit column remains consistent.
    endpoint_candidates = ["Earliest_metastatic_ER", "Metastatic_ER"]
    endpoint_source = next((c for c in endpoint_candidates if c in split.columns), None)
    if endpoint_source is None:
        raise ValueError(
            "Cohort_split.tsv must contain either 'Earliest_metastatic_ER' "
            "or 'Metastatic_ER'."
        )
    split = split.rename(columns={endpoint_source: "Metastatic_ER"})
    split["patient_id"] = split["patient_id"].map(normalize_patient_id)
    split["Primary_tumor_ER"] = split["Primary_tumor_ER"].map(normalize_er)
    split["Metastatic_ER"] = split["Metastatic_ER"].map(normalize_er)
    cohort = split.loc[
        split["Primary_tumor_ER"].eq("Positive")
        & split["Metastatic_ER"].isin(["Positive", "Negative"])
    ].copy()
    if cohort["patient_id"].duplicated().any():
        raise ValueError("The fixed cohort contains duplicate patient IDs; expected one row per patient.")
    cohort["ER_evolution_group"] = np.where(
        cohort["Metastatic_ER"].eq("Positive"), "ER retained", "ER loss"
    )
    if len(cohort) != 199:
        print(f"WARNING: cohort has {len(cohort)} patients, not the expected 199. Review source versions.")

    # First cfDNA / Griffin sample.
    first_draw = pd.read_csv(FIRSTDRAW_TSV, sep="\t", dtype={"patient_id": str})
    required_fd = {"sample", "draw_day", "TFx_pct"}
    missing = required_fd - set(first_draw.columns)
    if missing:
        raise ValueError(f"First-draw input is missing: {sorted(missing)}")
    if first_draw["sample"].duplicated().any():
        raise ValueError("First-draw input contains duplicate sample IDs.")
    keep_fd = ["sample", "draw_day", "TFx_pct"] + [
        c for c in ("ER", "ER_source_day", "ER_lag_days") if c in first_draw.columns
    ]
    audit = cohort.merge(first_draw[keep_fd], on="sample", how="left", validate="one_to_one")
    if audit["draw_day"].isna().any():
        unmatched = audit.loc[audit["draw_day"].isna(), ["patient_id", "sample"]]
        raise ValueError("Cohort samples missing from first-draw input:\n" + unmatched.to_string(index=False))
    audit = audit.rename(columns={
        "ER": "Blood_aligned_ER_at_firstNB",
        "ER_source_day": "Blood_aligned_ER_source_day",
        "ER_lag_days": "Blood_aligned_ER_lag_days",
    })
    for column in ("Blood_aligned_ER_at_firstNB", "Blood_aligned_ER_source_day", "Blood_aligned_ER_lag_days"):
        if column not in audit:
            audit[column] = np.nan
    audit["Blood_aligned_ER_at_firstNB"] = audit["Blood_aligned_ER_at_firstNB"].map(normalize_er)
    audit["draw_day"] = pd.to_numeric(audit["draw_day"], errors="coerce")

    # Longitudinal metastatic diagnoses and ER assessments.
    clinical_sheet = find_clinical_sheet(CLINICAL_XLSX)
    clinical = pd.read_excel(CLINICAL_XLSX, sheet_name=clinical_sheet)
    clinical["patient_id"] = clinical["patient_id"].map(normalize_patient_id)
    clinical["Days_From_Primary_Diagnosis"] = pd.to_numeric(
        clinical["Days_From_Primary_Diagnosis"], errors="coerce"
    )
    clinical["ER_Status_normalized"] = clinical["ER_Status"].map(normalize_er)
    met = clinical.loc[
        clinical["Category"].eq("Met_Diagnosis")
        & clinical["Days_From_Primary_Diagnosis"].notna()
    ].copy()

    timing_rows = []
    for patient_id in audit["patient_id"]:
        records = met.loc[met["patient_id"].eq(patient_id)].sort_values("Days_From_Primary_Diagnosis")
        if records.empty:
            timing_rows.append({"patient_id": patient_id, "n_Met_Diagnosis_rows": 0,
                                "n_informative_Met_ER_rows": 0,
                                "Met_ER_status_changes_across_assessments": "No"})
            continue
        earliest_day = records["Days_From_Primary_Diagnosis"].min()
        earliest_rows = records.loc[records["Days_From_Primary_Diagnosis"].eq(earliest_day)]
        informative = records.loc[records["ER_Status_normalized"].isin(["Positive", "Negative"])]
        result = {
            "patient_id": patient_id,
            "Earliest_Met_Diagnosis_day": earliest_day,
            "Earliest_Met_ER_at_that_day": "/".join(sorted(earliest_rows["ER_Status_normalized"].dropna().unique())) or np.nan,
            "Earliest_Met_sites": ", ".join(sorted(earliest_rows["Tissue_Site"].dropna().astype(str).unique())) or np.nan,
            "n_Met_Diagnosis_rows": len(records),
            "n_informative_Met_ER_rows": len(informative),
            "Met_ER_status_changes_across_assessments": "Yes" if informative["ER_Status_normalized"].nunique() > 1 else "No",
        }
        if not informative.empty:
            first = informative.iloc[0]
            result.update({
                "First_informative_Met_ER_day": first["Days_From_Primary_Diagnosis"],
                "First_informative_Met_ER_status": first["ER_Status_normalized"],
                "First_informative_Met_ER_site": first["Tissue_Site"],
                "All_informative_Met_ER_assessments": "; ".join(
                    f"{int(r.Days_From_Primary_Diagnosis)}:{r.ER_Status_normalized}" +
                    (f" ({r.Tissue_Site})" if pd.notna(r.Tissue_Site) else "")
                    for _, r in informative.iterrows()
                ),
            })
        else:
            result.update({"First_informative_Met_ER_day": np.nan,
                           "First_informative_Met_ER_status": np.nan,
                           "First_informative_Met_ER_site": np.nan,
                           "All_informative_Met_ER_assessments": ""})
        timing_rows.append(result)
    audit = audit.merge(pd.DataFrame(timing_rows), on="patient_id", how="left", validate="one_to_one")

    audit["Lag_NB_minus_earliestMet_days"] = audit["draw_day"] - audit["Earliest_Met_Diagnosis_day"]
    audit["Lag_NB_minus_ERdef_days"] = audit["draw_day"] - audit["First_informative_Met_ER_day"]
    for days in (30, 60, 90):
        audit[f"Within_{days}d_earliestMet"] = audit["Lag_NB_minus_earliestMet_days"].abs().le(days)
        audit[f"Within_{days}d_ERdef"] = audit["Lag_NB_minus_ERdef_days"].abs().le(days)
    audit["Endpoint_matches_first_informative_ER"] = audit["Metastatic_ER"].eq(audit["First_informative_Met_ER_status"])
    audit["Endpoint_vs_blood_aligned"] = np.select(
        [audit["Blood_aligned_ER_at_firstNB"].isna(),
         audit["Metastatic_ER"].eq(audit["Blood_aligned_ER_at_firstNB"])],
        ["No informative ER on/before NB", "Concordant"], default="Discordant"
    )

    def timing_qc(row):
        if not row["Endpoint_matches_first_informative_ER"]: return "REVIEW: endpoint mismatch"
        lag = row["Lag_NB_minus_ERdef_days"]
        if pd.isna(lag): return "REVIEW: missing ER-defining assessment"
        if lag < -30: return "REVIEW: NB >30d before ER-defining assessment"
        if lag < 0: return "NB before ER-defining assessment"
        if abs(lag) <= 30: return "Near ER-defining assessment (±30d)"
        if abs(lag) <= 60: return "Near ER-defining assessment (±60d)"
        if abs(lag) <= 90: return "Near ER-defining assessment (±90d)"
        return "Outside ±90d of ER-defining assessment"
    audit["Timing_QC"] = audit.apply(timing_qc, axis=1)

    window_rows = []
    for anchor, suffix in (("Earliest metastatic diagnosis", "earliestMet"), ("ER-defining metastatic assessment", "ERdef")):
        for days in (30, 60, 90):
            flag = audit[f"Within_{days}d_{suffix}"]
            retained = int((flag & audit["ER_evolution_group"].eq("ER retained")).sum())
            loss = int((flag & audit["ER_evolution_group"].eq("ER loss")).sum())
            window_rows.append({"Timing_anchor": anchor, "Window_days": f"±{days}",
                                "ER_retained_n": retained, "ER_loss_n": loss, "Total_n": retained + loss})
    windows = pd.DataFrame(window_rows)
    blood_summary = (audit.assign(Blood_aligned_display=audit["Blood_aligned_ER_at_firstNB"].fillna("No prior informative ER"))
                     .groupby(["Metastatic_ER", "Blood_aligned_display"], dropna=False).size().reset_index(name="n"))
    summary_rows = [{"Section": "Cohort", "Metric": "Total patients", "Value": len(audit)},
                    {"Section": "Cohort", "Metric": "ER retained", "Value": int(audit["ER_evolution_group"].eq("ER retained").sum())},
                    {"Section": "Cohort", "Metric": "ER loss", "Value": int(audit["ER_evolution_group"].eq("ER loss").sum())},
                    {"Section": "QC", "Metric": "Endpoint matches first informative metastatic ER", "Value": int(audit["Endpoint_matches_first_informative_ER"].sum())},
                    {"Section": "QC", "Metric": "Patients with metastatic ER status change", "Value": int(audit["Met_ER_status_changes_across_assessments"].eq("Yes").sum())},
                    {"Section": "QC", "Metric": "First NB before earliest metastatic diagnosis", "Value": int(audit["Lag_NB_minus_earliestMet_days"].lt(0).sum())}]
    for row in windows.itertuples(index=False):
        for label, value in (("ER retained", row.ER_retained_n), ("ER loss", row.ER_loss_n), ("total", row.Total_n)):
            summary_rows.append({"Section": "Timing windows", "Metric": f"{row.Timing_anchor} {row.Window_days} {label}", "Value": value})
    summary = pd.DataFrame(summary_rows)

    group_order = audit["ER_evolution_group"].map({"ER loss": 0, "ER retained": 1})
    audit = audit.assign(_group_order=group_order).sort_values(["_group_order", "patient_id", "sample"]).drop(columns="_group_order")
    audit.to_csv(AUDIT_TSV, sep="\t", index=False)
    summary.to_csv(SUMMARY_TSV, sep="\t", index=False)
    with pd.ExcelWriter(AUDIT_XLSX) as writer:
        summary.to_excel(writer, sheet_name="Summary", index=False)
        windows.to_excel(writer, sheet_name="Timing_windows", index=False)
        blood_summary.to_excel(writer, sheet_name="Blood_aligned_ER", index=False)
        audit.to_excel(writer, sheet_name="Audit_199", index=False)
    print(f"Clinical sheet: {clinical_sheet}")
    print(audit["ER_evolution_group"].value_counts().to_string())
    print("\nSaved:\n" + "\n".join(map(str, (AUDIT_TSV, SUMMARY_TSV, AUDIT_XLSX))))


if __name__ == "__main__":
    main()

Clinical sheet: AUP_ClinicalData_20260609.NO.DA
ER retained    179
ER loss         20

Saved:
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Griffin/Downstream/ER_phenotype/199_patient_ER_timing_audit/AURORA_199_patient_ER_timing_audit.tsv
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Griffin/Downstream/ER_phenotype/199_patient_ER_timing_audit/AURORA_199_patient_ER_timing_summary.tsv
/ix1/alee/LO_LAB/Personal/Rahul/cfDNA_project/AURORA/Griffin/Downstream/ER_phenotype/199_patient_ER_timing_audit/AURORA_199_patient_ER_timing_audit.xlsx
